In [1]:
%load_ext autoreload
%autoreload 2

import os,sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import util as yu
from util import *
import util_charge as yuc

yu.setpath('analysis_3pt_laplace')
ens='b'
enss=[ens]
tfs=[8,10,12,14,16,18,20]
ylim_global=[0,80]

In [2]:
[c2ptM,tf2c3ptM,c2ptCorrDic_NJN]=yu.load_pkl_reg('data',pathlabel='processData')
tf2c3ptM={tf:np.real(tf2c3ptM[tf]) for tf in tf2c3ptM.keys()}

t=yu.load_pkl_reg('ens2pars_jk_meffnst_selected',pathlabel='analysis_2pt')
[pars_jk_meff1st,pars_jk_meff2st,pars_jk_meff3st]=[t[0][ens],t[1][ens],t[2][ens]]

In [6]:
g='gS+'
ds=1
symmetrizeQ=True

app = {1:'',2:'_ds2'}[ds]

ens2dics={}
for ens in enss:
    c2pt=c2ptM[:,:,0,0]
    tf2c2pt=c2ptCorrDic_NJN
    tf2c3pt={tf:tf2c3ptM[tf][:,:,0,0] for tf in tfs}
    tf2ratio={tf:tf2c3ptM[tf][:,:,0,0]/tf2c2pt[tf][:,None] for tf in tfs}
    
    tfmins_2st=tfs[:-3]
    # tcmins_2st=np.arange(1,int(0.5/yu.ens2a[ens]),int(0.2/yu.ens2a[ens]))
    tcmins_2st=np.arange(1,round(0.45/yu.ens2a[ens]))
    
    label2fits={}
    for label in ['2st2step_SYM','2st2step_SYMshare','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']:
        label2fits[label]=yu.doFits_3pt(label,tf2ratio,tfmins_2st,tcmins_2st,pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,downSampling=[1,ds],label=f'{g}_{ens}_{label}{app}_{symmetrizeQ}',verbose=1)
        
    # n=int(0.2/yu.ens2a[ens])
    n=2
    def dE2lbd(dE):
        lbd0=dE*n
        return np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)
    
    def lbd2tf2ratio(dE):    
        lbd=dE2lbd(dE)
        
        tf2ratio={}
        for tf in tfs:
            c3=tf2c3pt[tf]
            c3=-(np.roll(c3,-n,axis=-1)+np.roll(c3,n,axis=-1)-2*c3) + lbd**2*c3
            c2=(lbd**2)*tf2c2pt[tf]
            tf2ratio[tf]=c3/c2[:,None]
        return tf2ratio
    tfmins=tfmins_2st
    tcmins=[n+tcmin for tcmin in tcmins_2st]
    
    fits_laplace=yu.doFits_3pt_lbd(lbd2tf2ratio,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'{g}_{ens}_lbd{app}_{symmetrizeQ}',verbose=1,downSampling=[1,ds],overwrite=False)
    fits_laplace=[[(tfmin,tcmin),np.array([pars_jk[:,0],np.abs(pars_jk[:,1])]).T,chi2_jk,Ndof] for (tfmin,tcmin),pars_jk,chi2_jk,Ndof in fits_laplace]

    fitlabel_chosen=(8,n+2)
    # fitlabel_chosen=fits_laplace_ds2[0][0]
    fit_MA_laplace=yu.doMA_3pt(fits_laplace,fitlabels=fitlabel_chosen)
    print(ens,fitlabel_chosen,yu.jackme_un2str(fit_MA_laplace[0][:,1]*yu.ens2aInv[ens]))

    dE=np.mean(fit_MA_laplace[0][:,1])
    tf2ratio_laplace=lbd2tf2ratio(dE)
    # fits_const_2=yu.doFits_3pt('const',tf2ratio_laplace,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'const_2_laplace'+extraLabel)
    # fit_const_MA_2=yu.doMA_3pt(fits_const_2,fitlabels=fitlabel_chosen)

    xunit=yu.ens2a[ens]
    yunit=yu.ens2amul_iso[ens]*yu.ens2aInv[ens]
    
    dic={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,None,None,None,None],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
        'xyunit':[xunit,yunit],
        'mfc:[global]':['None'],
    }
    dic_lbd={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio_laplace,None,fits_laplace,None,None],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,n+1,None],
        'xyunit':[xunit,yunit],
        'mfc:[global]':['white'],
    }
    
    label2dic={}
    for label in label2fits.keys():
        label2dic[label]={
            'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,None,label2fits[label]],
            'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
            'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,5,None,None],
            'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
            'xyunit':[xunit,yunit],
            'mfc:[global]':['None'],
        }
    
    ens2dics[ens]=[dic,dic_lbd,label2dic]

colHeaders=['ratio','fit_laplace','2st2step_SYM','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']
fig,axs=yu.makePlot_3pt([ens2dics[ens][0] for ens in enss],shows=['rainbow'],colHeaders=colHeaders,fontsize_colHeaders=20, sharey=True)
yu.addRowHeader(axs,[yu.ens2label[ens] for ens in enss])
axs[0,0].set_ylim([0,80])
fig,axs=yu.makePlot_3pt([ens2dics[ens][1] for ens in enss],shows=['rainbow']+[None]*(colHeaders.index('fit_laplace')-1)+['fit_const'],figAxs=(fig,axs),colHeaders=None)

for i,label in enumerate(colHeaders):
    if label in label2dic.keys(): 
        fig,axs=yu.makePlot_3pt([ens2dics[ens][2][label] for ens in enss],shows=['rainbow']+[None]*(i-1)+['fit_2st'],figAxs=(fig,axs),colHeaders=None)
fig,axs=yu.makePlot_3pt([ens2dics[ens][2]['2st2step_SYMshare'] for ens in enss],shows=['rainbow','fit_2st'],figAxs=(fig,axs),colHeaders=None)
        
yu.finalizePlot(f'rainbow_fits{app}')

b (8, 4) 364(25)
